In [ ]:
import os
import ipywidgets as widgets
from IPython.display import display

bag_paths = [
    "./data/dataset_path",
    "./data/dataset_path",
    "./data/dataset_path",
    "./data/dataset_path",
    "./data/dataset_path",
    "../../content/realdata/data/smallpebbleslegro_2025-06-10-12-48-00.bag,"
    "../../content/realdata/concrete_legro_2025-06-09-14-23-57.bag",
    "../../content/realdata/concrete_legro_2025-06-10-12-11-34.orig.bag",
    "../../content/realdata/ramp_legro_2025-06-09-14-10-54.bag",
    "../../content/realdata/grass_legro_2025-06-09-14-33-00.bag",
    "../../content/realdata/asphalt_outra_legro_2025-06-09-14-30-51.bag",
    "../../content/realdata/data/legro_2025-06-05-16-48-33.bag",
    "../../content/realdata/data/legro_2025-06-05-16-53-58.bag",
    "./data/dataset_path",
    "./data/dataset_path",
    "./data/dataset_path",
    "./data/dataset_path",
    "./data/dataset_path",
    "./data/dataset_path",
    "./data/dataset_path",
    "./data/dataset_path",
    "./data/dataset_path",
    "./data/dataset_path" 
]

list_widget = []
bag_paths_selected = []

for bag_path in bag_paths:
    if not os.path.exists(bag_path):
        # print(f"File not found: {bag_path}")
        None
    else:
        list_widget.append(
            widgets.Checkbox(value=False, description=bag_path,layout=widgets.Layout(width='100%')))
        

container = widgets.VBox(list_widget)

display(container)

btn = widgets.Button(description="Load bag(s)")
out = widgets.Output()
display(btn, out)

def on_click(b):
    with out:
        out.clear_output()
        selected = [cb.description for cb in list_widget if cb.value]
        print("Selected:", selected)
        global bag_paths_selected
        bag_paths_selected = selected

btn.on_click(on_click)

In [ ]:
import load_data

required_topics = ["/estimated_robot_state", "odom", "joint_state", "/unitree/contact_state", "/unitree/imu", "/unitree/raw_contact"]

load_data.ros1_check_file(bag_paths_selected,required_topics)

In [ ]:
pip install mcap mcap-ros2-support

In [ ]:
from mcap.reader import make_reader
from mcap_ros2.decoder import DecoderFactory
import os

mcap_path = "./data/dataset_path"

required_topics = [
    "/estimated_robot_state", "odom", "joint_state",
    "/unitree/contact_state", "/unitree/imu", "/unitree/raw_contact"
]

if not os.path.exists(mcap_path):
    print("File does not exist.")
else:
    with open(mcap_path, "rb") as f:
        reader = make_reader(f, decoder_factories=[DecoderFactory()])
        
        # Get all topics in the file
        summary = reader.get_summary()
        topics_found = {ch.topic for ch in summary.channels.values()}
        print("Topics found:", topics_found)
        
        for topic in required_topics:
            if topic in topics_found:
                print(f"  ✓ {topic} found")
            else:
                print(f"  ✗ {topic} NOT found")
                

with open(mcap_path, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(
        topics=["/unitree/imu"]
    ):
        print(f"Topic: {channel.topic}")
        print(f"Message: {ros_msg}")
        break  # Remove to read all messages

In [ ]:
from mcap.reader import make_reader
from mcap_ros2.decoder import DecoderFactory
import os

mcap_path = "./data/dataset_path"

with open(mcap_path, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    
    print("=" * 60)
    print("SCHEMA INFO (message field definitions)")
    print("=" * 60)
    summary = reader.get_summary()
    for schema_id, schema in summary.schemas.items():
        print(f"\nSchema: {schema.name}")
        if schema.data:
            print(schema.data.decode("utf-8", errors="ignore"))

with open(mcap_path, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    
    print("\n" + "=" * 60)
    print("SAMPLE MESSAGES FROM /lowstate")
    print("=" * 60)
    count = 0
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(topics=["/lowstate"]):
        print(f"\n--- Message {count+1} ---")
        print(ros_msg)
        count += 1
        if count >= 3:
            break

with open(mcap_path, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    
    print("\n" + "=" * 60)
    print("SAMPLE MESSAGES FROM /sportmodestate")
    print("=" * 60)
    count = 0
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(topics=["/sportmodestate"]):
        print(f"\n--- Message {count+1} ---")
        print(ros_msg)
        count += 1
        if count >= 3:
            break

In [ ]:
# THIS HERE SHOULD EXTRACT THE CONTACT STATE DATA FROM /lowstate AND SAVE TO CSV FOR ANALYSIS
from mcap.reader import make_reader
from mcap_ros2.decoder import DecoderFactory
import pandas as pd
import matplotlib.pyplot as plt
import os

mcap_path = "./data/dataset_path"

# ── Extract contact-related data from /lowstate ──────────────────────────────
rows = []

with open(mcap_path, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(
        topics=["/lowstate"]
    ):
        ms = ros_msg.motor_state

        row = {
            "tick": ros_msg.tick,

            # ── Raw foot force sensors (equivalent to /unitree/raw_contact) ──
            "foot_force_FR": ros_msg.foot_force[0],
            "foot_force_FL": ros_msg.foot_force[1],
            "foot_force_RR": ros_msg.foot_force[2],
            "foot_force_RL": ros_msg.foot_force[3],

            # ── Estimated foot force ──────────────────────────────────────────
            "foot_force_est_FR": ros_msg.foot_force_est[0],
            "foot_force_est_FL": ros_msg.foot_force_est[1],
            "foot_force_est_RR": ros_msg.foot_force_est[2],
            "foot_force_est_RL": ros_msg.foot_force_est[3],

            # ── Binary contact state (True if force above threshold) ──────────
            "contact_FR": int(ros_msg.foot_force[0] > 20),
            "contact_FL": int(ros_msg.foot_force[1] > 20),
            "contact_RR": int(ros_msg.foot_force[2] > 20),
            "contact_RL": int(ros_msg.foot_force[3] > 20),

            # ── Joint torques per leg (useful for contact estimation) ─────────
            "tau_FR_calf": ms[2].tau_est,
            "tau_FL_calf": ms[5].tau_est,
            "tau_RR_calf": ms[8].tau_est,
            "tau_RL_calf": ms[11].tau_est,
        }
        rows.append(row)

df = pd.DataFrame(rows)

# ── Normalize tick to start at 0 ─────────────────────────────────────────────
df["tick"] = df["tick"] - df["tick"].iloc[0]

print("Columns:", df.columns.tolist())
print("\nStats per column:")
print(df.describe())
print("\nFirst few rows:")
print(df.head())

# ── Save CSV ──────────────────────────────────────────────────────────────────
output_dir = "./data/dataset_path"
os.makedirs(output_dir, exist_ok=True)
base_name = os.path.splitext(os.path.basename(mcap_path))[0]
csv_path = os.path.join(output_dir, f"{base_name}_contact.csv")
df.to_csv(csv_path, index=False)
print(f"\nSaved: {csv_path}")

# ── Plot raw foot forces ──────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Raw foot forces
ax = axes[0]
for leg, color in zip(["FR", "FL", "RR", "RL"], ["tab:blue", "tab:orange", "tab:green", "tab:red"]):
    ax.plot(df["tick"], df[f"foot_force_{leg}"], label=leg, color=color)
ax.set_ylabel("Raw Foot Force")
ax.set_title("Raw Foot Forces (Go2 /lowstate)")
ax.legend()
ax.grid(True)

# Binary contact state
ax = axes[1]
for i, (leg, color) in enumerate(zip(["FR", "FL", "RR", "RL"], ["tab:blue", "tab:orange", "tab:green", "tab:red"])):
    ax.plot(df["tick"], df[f"contact_{leg}"] + i * 1.2, label=leg, color=color)
ax.set_ylabel("Contact (0/1, shifted)")
ax.set_title("Binary Contact State (threshold > 20)")
ax.legend()
ax.grid(True)

# Calf torques
ax = axes[2]
for leg, color in zip(["FR", "FL", "RR", "RL"], ["tab:blue", "tab:orange", "tab:green", "tab:red"]):
    ax.plot(df["tick"], df[f"tau_{leg}_calf"], label=leg, color=color)
ax.set_ylabel("Calf Torque [Nm]")
ax.set_title("Calf Joint Torques (contact proxy)")
ax.set_xlabel("Tick")
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, f"{base_name}_contact.png"), dpi=150)
plt.show()
print("Plot saved.")

In [ ]:
from mcap.reader import make_reader
from mcap_ros2.decoder import DecoderFactory
import pandas as pd
import numpy as np
import os

mcap_path = "./data/dataset_path"
output_dir = "./data/dataset_path"
os.makedirs(output_dir, exist_ok=True)

# ── Go2 Kinematics (body frame, meters) ──────────────────────────────────────
# Hip offsets from body center
HIP_OFFSETS = {
    "FR": np.array([ 0.1934,  -0.0465, 0.0]),
    "FL": np.array([ 0.1934,   0.0465, 0.0]),
    "RR": np.array([-0.1934,  -0.0465, 0.0]),
    "RL": np.array([-0.1934,   0.0465, 0.0]),
}
L_THIGH = 0.213   # thigh link length [m]
L_CALF  = 0.213   # calf link length [m]
L_HIP   = 0.0955  # hip lateral offset [m]
SIDE    = {"FR": -1, "FL": 1, "RR": -1, "RL": 1}  # right=-1, left=+1

def forward_kinematics(hip_q, thigh_q, calf_q, leg):
    """Returns EE position in body frame for one leg."""
    s = SIDE[leg]
    offset = HIP_OFFSETS[leg]

    # Hip rotation (abduction/adduction)
    hip_lateral = s * L_HIP
    p_hip = np.array([
        0.0,
        hip_lateral * np.cos(hip_q),
        hip_lateral * np.sin(hip_q)
    ])

    # Thigh rotation
    p_thigh = np.array([
        0.0,
        -L_THIGH * np.sin(thigh_q),
        -L_THIGH * np.cos(thigh_q)
    ])

    # Calf rotation (relative to thigh)
    p_calf = np.array([
        0.0,
        -L_CALF * np.sin(thigh_q + calf_q),
        -L_CALF * np.cos(thigh_q + calf_q)
    ])

    ee_pos = offset + p_hip + p_thigh + p_calf
    return ee_pos

# ── Joint index map (motor_state indices) ────────────────────────────────────
JOINTS = {
    "FR": (0, 1, 2),
    "FL": (3, 4, 5),
    "RR": (6, 7, 8),
    "RL": (9, 10, 11),
}

rows = []

with open(mcap_path, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    prev_ee = {leg: None for leg in ["FR", "FL", "RR", "RL"]}
    prev_tick = None

    for schema, channel, message, ros_msg in reader.iter_decoded_messages(
        topics=["/lowstate"]
    ):
        ms   = ros_msg.motor_state
        imu  = ros_msg.imu_state
        tick = ros_msg.tick

        row = {
            "tick": tick,

            # ── IMU - body frame ──────────────────────────────────────────────
            # Linear acceleration (body frame) [m/s²]
            "acc_x":   imu.accelerometer[0],
            "acc_y":   imu.accelerometer[1],
            "acc_z":   imu.accelerometer[2],
            # Angular velocity (body frame) [rad/s]
            "gyro_x":  imu.gyroscope[0],
            "gyro_y":  imu.gyroscope[1],
            "gyro_z":  imu.gyroscope[2],
            # Orientation
            "roll":    imu.rpy[0],
            "pitch":   imu.rpy[1],
            "yaw":     imu.rpy[2],
            "quat_x":  imu.quaternion[0],
            "quat_y":  imu.quaternion[1],
            "quat_z":  imu.quaternion[2],
            "quat_w":  imu.quaternion[3],

            # ── Contact state ─────────────────────────────────────────────────
            "contact_FR": int(ros_msg.foot_force[0] > 20),
            "contact_FL": int(ros_msg.foot_force[1] > 20),
            "contact_RR": int(ros_msg.foot_force[2] > 20),
            "contact_RL": int(ros_msg.foot_force[3] > 20),
            "foot_force_FR": ros_msg.foot_force[0],
            "foot_force_FL": ros_msg.foot_force[1],
            "foot_force_RR": ros_msg.foot_force[2],
            "foot_force_RL": ros_msg.foot_force[3],
        }

        # ── Joint positions & velocities ──────────────────────────────────────
        for leg, (hi, ti, ci) in JOINTS.items():
            row[f"q_{leg}_hip"]   = ms[hi].q
            row[f"q_{leg}_thigh"] = ms[ti].q
            row[f"q_{leg}_calf"]  = ms[ci].q
            row[f"dq_{leg}_hip"]  = ms[hi].dq
            row[f"dq_{leg}_thigh"]= ms[ti].dq
            row[f"dq_{leg}_calf"] = ms[ci].dq
            row[f"tau_{leg}_hip"]   = ms[hi].tau_est
            row[f"tau_{leg}_thigh"] = ms[ti].tau_est
            row[f"tau_{leg}_calf"]  = ms[ci].tau_est

        # ── EE position via Forward Kinematics (body frame) ───────────────────
        dt = (tick - prev_tick) / 1000.0 if prev_tick is not None else None

        for leg, (hi, ti, ci) in JOINTS.items():
            ee_pos = forward_kinematics(
                ms[hi].q, ms[ti].q, ms[ci].q, leg
            )
            row[f"ee_pos_{leg}_x"] = ee_pos[0]
            row[f"ee_pos_{leg}_y"] = ee_pos[1]
            row[f"ee_pos_{leg}_z"] = ee_pos[2]

            # EE velocity: finite difference of FK positions
            if prev_ee[leg] is not None and dt and dt > 0:
                ee_vel = (ee_pos - prev_ee[leg]) / dt
            else:
                ee_vel = np.array([0.0, 0.0, 0.0])

            row[f"ee_vel_{leg}_x"] = ee_vel[0]
            row[f"ee_vel_{leg}_y"] = ee_vel[1]
            row[f"ee_vel_{leg}_z"] = ee_vel[2]

            prev_ee[leg] = ee_pos

        prev_tick = tick
        rows.append(row)

# ── Save ──────────────────────────────────────────────────────────────────────
df = pd.DataFrame(rows)
df["tick"] = df["tick"] - df["tick"].iloc[0]

base_name = os.path.splitext(os.path.basename(mcap_path))[0]
out_path = os.path.join(output_dir, f"{base_name}_full.csv")
df.to_csv(out_path, index=False)

print(f"Saved: {out_path}  ({len(df)} rows, {len(df.columns)} columns)")
print("\nColumns:", df.columns.tolist())
print("\nEE position sample (FR leg, body frame):")
print(df[["tick", "ee_pos_FR_x", "ee_pos_FR_y", "ee_pos_FR_z"]].head(5))
print("\nEE velocity sample (FR leg):")
print(df[["tick", "ee_vel_FR_x", "ee_vel_FR_y", "ee_vel_FR_z"]].head(5))
print("\nIMU sample:")
print(df[["tick", "acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]].head(5))

In [ ]:
#SIGNAL PLOT PER LEG
import importlib
importlib.reload(load_data)

df_raw = load_data.ros1_load_topic(bag_paths_selected[0], "/unitree/contact_state")

contact_cols = ['ee_state_0', 'ee_state_1', 'ee_state_2', 'ee_state_3']

plt.figure(figsize=(14,6))

for col in contact_cols:
    plt.plot(df_raw["Time"], df_raw[col], label=col)

plt.xlabel("Time [s]")
plt.ylabel("Raw Contact Sensor Reading [a.u.]")
plt.title("Unitree Raw Contact Signals (per leg)")
plt.legend(title="Legs")
plt.grid(True, alpha=0.3)
plt.show()

# Create subplots (4 rows, 1 column)
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

for i, col in enumerate(contact_cols):
    axes[i].plot(df_raw["Time"], df_raw[col], color="C{}".format(i))
    axes[i].set_ylabel("Raw Value [a.u.]")
    axes[i].set_title(f"{col} Contact Signal")
    axes[i].grid(True, alpha=0.3)
    # Example: zoom into 10–20 seconds
    #axes[i].set_xlim(10, 200)

# Label the shared x-axis
axes[-1].set_xlabel("Time [s]")

plt.tight_layout()
plt.show()


print("Columns in raw contact df:", df_raw.columns)
print(df_raw.head())#

# Quick stats
print("\nStats per column:")
print(df_raw.describe())

# Peek at first few rows
print("\nFirst few rows:")
print(df_raw.head())

In [ ]:
bag_file = bag_paths_selected[0]

import load_config
load_config.load_config()


In [ ]:

# -------------------------------------------------------------------------------------------------------------------------------------
# This part here does Otsu Thresholding of the missed contact states, and creates improved contact labels based on that. It also 
# visualizes the results and saves the improved labels for ML training.
# -------------------------------------------------------------------------------------------------------------------------------------
bag_file = bag_paths_selected[0]


import bagpy
from bagpy import bagreader
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

# Load the data (assuming you already have this)
#bag_file = "./data/dataset_path"
bag = bagreader(bag_file)
csvfile_raw = bag.message_by_topic("/unitree/contact_state")
df_raw = pd.read_csv(csvfile_raw)

# Contact columns
contact_cols = ['raw_contacts_0', 'raw_contacts_1', 'raw_contacts_2', 'raw_contacts_3']
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

# Function to detect optimal threshold using multiple methods
def analyze_contact_thresholds(data, col_name):
    """Analyze data to suggest optimal contact detection threshold
       So we do a thorough analysing of the data here"""

    # Method 1: Statistical analysis
    mean_val = np.mean(data)
    std_val = np.std(data)
    median_val = np.median(data)

    # Method 2: Histogram-based (find valley between peaks if bimodal)
    hist, bin_edges = np.histogram(data, bins=50)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    # Method 3: Otsu's method (adapted for threshold selection)
    def otsu_threshold(data):
        hist, bin_edges = np.histogram(data, bins=256)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

        # Normalize histogram
        hist = hist.astype(float) / hist.sum()

        max_variance = 0
        best_threshold = 0

        for i in range(1, len(hist)-1):
            # Background and foreground weights
            w0 = hist[:i].sum()
            w1 = hist[i:].sum()

            if w0 == 0 or w1 == 0:
                continue

            # Background and foreground means
            mu0 = (hist[:i] * bin_centers[:i]).sum() / w0 if w0 > 0 else 0
            mu1 = (hist[i:] * bin_centers[i:]).sum() / w1 if w1 > 0 else 0

            # Between-class variance
            variance = w0 * w1 * (mu0 - mu1)**2

            if variance > max_variance:
                max_variance = variance
                best_threshold = bin_centers[i]

        return best_threshold

    otsu_thresh = otsu_threshold(data)

    print(f"\n=== Threshold Analysis for {col_name} ===")
    print(f"Mean: {mean_val:.1f}")
    print(f"Median: {median_val:.1f}")
    print(f"Std: {std_val:.1f}")
    print(f"Mean + 0.5*Std: {mean_val + 0.5*std_val:.1f}")
    print(f"Mean + 1*Std: {mean_val + std_val:.1f}")
    print(f"Otsu threshold: {otsu_thresh:.1f}")
    print(f"Data range: {np.min(data):.1f} to {np.max(data):.1f}")

    return {
        'mean': mean_val,
        'median': median_val,
        'std': std_val,
        'mean_plus_half_std': mean_val + 0.5*std_val,
        'mean_plus_std': mean_val + std_val,
        'otsu': otsu_thresh
    }

# Analyze each sensor
threshold_analysis = {}
for col in contact_cols:
    threshold_analysis[col] = analyze_contact_thresholds(df_raw[col], col)

# Manual threshold setting based on analysis
# You can adjust these based on the analysis output
default_thresholds = {
    'raw_contacts_0': 300 - 100,  # Adjust based on analysis
    'raw_contacts_1': 500 - 150,  # Adjust based on analysis
    'raw_contacts_2': 100 - 50,  # Adjust based on analysis
    'raw_contacts_3': 400 - 100,  # Adjust based on analysis (higher due to sensitivity)
}


# Function to apply threshold-based contact detection
def detect_contacts_threshold(data, threshold, hysteresis_factor=0.9, min_contact_duration=0.01):
    """
    Apply threshold-based contact detection with hysteresis and minimum duration filtering

    Parameters:
    - data: raw sensor data
    - threshold: contact detection threshold
    - hysteresis_factor: factor for lower threshold (prevents chattering)
    - Common “rule of thumb” hysterisis of 5–15% below the main threshold usually works well for sensors that are moderately noisy.
    - min_contact_duration: minimum contact duration in seconds (1000Hz sampling)
    """

    # Hysteresis thresholds
    upper_thresh = threshold
    lower_thresh = threshold * hysteresis_factor

    # Initialize contact state
    contacts = np.zeros(len(data), dtype=bool)
    current_state = False

    # Apply hysteresis
    for i in range(len(data)):
        if not current_state and data[i] > upper_thresh:
            current_state = True
        elif current_state and data[i] < lower_thresh:
            current_state = False
        contacts[i] = current_state

    # Apply minimum duration filtering (remove very short contacts/gaps)
    # At500Hz sampling rate
    min_samples = int(min_contact_duration * 500)

    # Remove short contact periods
    contact_diff = np.diff(contacts.astype(int))
    contact_starts = np.where(contact_diff == 1)[0] + 1
    contact_ends = np.where(contact_diff == -1)[0] + 1

    # Handle edge cases
    if contacts[0]:
        contact_starts = np.insert(contact_starts, 0, 0)
    if contacts[-1]:
        contact_ends = np.append(contact_ends, len(contacts))


    # Filter short contacts
    ls,le = None,None
    for start, end in zip(contact_starts, contact_ends):

        if ls and le:
            if start - le < min_samples:
                contacts[le:start] = True
            pass

        if end - start < min_samples:
            ls,le = start,end
            #contacts[start:end] = False

    return contacts

# Apply threshold-based detection
print("Applying threshold-based contact detection...")
detected_contacts = {}

for col in contact_cols:
    # Use suggested threshold or manual override
    threshold = default_thresholds[col]

    # Detect contacts
    detected_contacts[col] = detect_contacts_threshold(
        df_raw[col],
        threshold=threshold,
        hysteresis_factor=0.9,
        min_contact_duration=0.05  # 50ms minimum contact
    )

    # Calculate contact statistics
    contact_percentage = np.mean(detected_contacts[col]) * 100
    print(f"{col}: Threshold={threshold:.1f}, Contact={contact_percentage:.1f}% of time")

# Create visualization comparing original vs threshold-based detection
fig, axes = plt.subplots(4, 1, figsize=(16*2, 12*2), sharex=True)

for i, col in enumerate(contact_cols):

    # Plot raw signal
    axes[i].plot(df_raw["Time"], df_raw[col], color=colors[i], alpha=0.7, linewidth=1, label='Raw Signal')

    # Plot threshold line
    axes[i].axhline(y=default_thresholds[col], color='red', linestyle='--', alpha=0.8, label=f'Threshold ({default_thresholds[col]})')

    axes[i].axhline(y=default_thresholds[col]*0.9, color='red', linestyle='--', alpha=0.8, label=f'Threshold ({default_thresholds[col]})')

    # Create twin axis for contact states
    ax2 = axes[i].twinx()


    # Plot threshold-based detection
    #ax2.plot(df_raw["Time"], detected_contacts[col].astype(int),
    #        color='darkgreen', linewidth=2, alpha=0.8)

    ax2.fill_between(df_raw["Time"], 0, detected_contacts[col].astype(int),
                    alpha=0.2, color='green', label='Threshold-Based Contact')

    ax2.set_ylim(-0.1, 1.3)
    ax2.set_ylabel('Contact State', fontsize=10)
    ax2.tick_params(axis='y', labelsize=8)

    # Format main axis
    axes[i].set_ylabel("Raw Value [a.u.]", fontsize=10)
    axes[i].set_title(f"{col} - Threshold-Based Contact Detection", fontsize=12)
    axes[i].grid(True, alpha=0.3)
    axes[i].tick_params(axis='y', labelsize=8)

    # Combine legends
    lines1, labels1 = axes[i].get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    axes[i].legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)

axes[-1].set_xlabel("Time [s]", fontsize=12)
plt.tight_layout()
plt.show()

# Save the improved contact states/unitree/contact_state
contact_df = df_raw[['Time'] + contact_cols].copy()
for col in contact_cols:
    contact_df[f'{col}_contact_threshold'] = detected_contacts[col].astype(int)

print(f"\nContact detection complete!")
print(f"New dataframe shape: {contact_df.shape}")
print(f"Columns: {contact_df.columns.tolist()}")

# Export for ML training
contact_df.to_csv('concrete_improved_contact_labels.csv', index=False)
print("Saved improved contact labels to 'concrete_improved_contact_labels.csv'")

# Display sample of the results
print("\nSample of improved contact labels:")
print(contact_df[['Time', 'raw_contacts_0', 'raw_contacts_0_contact_threshold',
                 'raw_contacts_3', 'raw_contacts_3_contact_threshold']].head(10))

In [ ]:
# --------------------------------------------------------------------------------------------------------------------------------------
# we can now use the improved contact states or theraw contact states for training. The improved contact states are more accurate and 
# less noisy, but the raw contact states might contain more information that the model can learn from. 
# We can experiment with both and see which one gives better results in terms of model performance.
# --------------------------------------------------------------------------------------------------------------------------------------
import bagpy
from bagpy import bagreader
import pandas as pd
import matplotlib.pyplot as plt

def analyze_contact_data(bag_file):
    """
    Convert ROS bag contact topics to CSV and read the data.
    """

    # Initialize bag reader
    bag = bagreader(bag_file)

    print("Converting topics to CSV...")

    # Convert raw contact data to CSV
    try:
        raw_contact_csv = bag.message_by_topic('/unitree/contact_state')
        print(f"✓ Contact state data converted: {raw_contact_csv}")
    except Exception as e:
        print(f"✗ Failed to convert /unitree/contact_state: {e}")
        return None

    # Read the CSV file
    print("\nReading CSV file...")
    raw_contact_df = pd.read_csv(raw_contact_csv)

    # Save CSV copy (DO NOT overwrite the bag file)
    output_csv = "./data/dataset_path"
    raw_contact_df.to_csv(output_csv, index=False)

    print("✓ Contact state data read into DataFrame")
    print(f"Contact state data shape: {raw_contact_df.shape}")
    
    print("\nUnique values per leg:")
    for leg in ['ee_state_0','ee_state_1','ee_state_2','ee_state_3']:
        print(leg, raw_contact_df[leg].unique())

    # Display column information
    print("\n=== CONTACT STATE DATA COLUMNS ===")
    print(raw_contact_df.columns.tolist())

    print("\nFirst few rows of contact state data:")
    print(raw_contact_df.head())

    return raw_contact_df

def plot_contact_state(raw_contact_df, time_window=None):
    """
    Plot raw contact state (ee_state) over time for 4 legs
    """

    if raw_contact_df is None:
        print("No data to plot!")
        return

    # Convert timestamps
    if 'Time' in raw_contact_df.columns:
        raw_contact_df['timestamp'] = pd.to_numeric(raw_contact_df['Time'])
        raw_contact_df['time_sec'] = raw_contact_df['timestamp'] - raw_contact_df['timestamp'].min()
    else:
        print("Time column not found!")
        return

    # Apply time window if specified
    if time_window:
        start_time, end_time = time_window
        raw_contact_df = raw_contact_df[
            (raw_contact_df['time_sec'] >= start_time) &
            (raw_contact_df['time_sec'] <= end_time)
        ]

    # Map contact states to legs
    raw_contact_df['FL'] = raw_contact_df['ee_state_0']
    raw_contact_df['FR'] = raw_contact_df['ee_state_1']
    raw_contact_df['BL'] = raw_contact_df['ee_state_2']
    raw_contact_df['BR'] = raw_contact_df['ee_state_3']

    legs = ['FL', 'FR', 'BL', 'BR']

    # Create plots
    fig, axes = plt.subplots(4, 1, figsize=(15, 10), sharex=True)
    fig.suptitle('Raw Contact States Over Time (Separate Legs)', fontsize=16, fontweight='bold')

    for i, leg in enumerate(legs):
        #axes[i].plot(raw_contact_df['time_sec'], raw_contact_df[leg], linewidth=2)
        axes[i].step(raw_contact_df['time_sec'], raw_contact_df[leg], where="post")
        axes[i].set_ylabel(f'{leg} Contact')
        axes[i].set_ylim(-0.1, 1.1)
        axes[i].grid(True, alpha=0.3)

    axes[-1].set_xlabel('Time (seconds)')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


def main(bag_file, time_window=None):
    """
    Main function to analyze contact data
    """

    print("Starting contact data analysis...")
    print(f"Bag file: {bag_file}")

    contact_df = analyze_contact_data(bag_file)

    if contact_df is not None:
        print("\n" + "=" * 50)
        print("Creating visualizations...")
        plot_contact_state(contact_df, time_window)
    else:
        print("Failed to read contact data!")


if __name__ == "__main__":

    bag_file = "./data/dataset_path"

    #main(bag_file)
    main(bag_file, time_window=(10, 15))

    # Optional example:
    # main(bag_file, time_window=(10, 60))


In [ ]:
# --------------------------------------------------------------------------------------------------------------------------------------
# THIS IS WHERE THRESHOLDING BEGINS, IF YOUR CONTACT STATES DATA IS AYT THEN YOU DONT NEED IT, BUT IF YOUR CONTACT STATES DATA IS NOISY 
# AND HAS MISSED CONTACTS THEN THIS CAN HELP YOU IMPROVE THE LABELS FOR TRAINING.
# --------------------------------------------------------------------------------------------------------------------------------------
import bagpy
from bagpy import bagreader
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


# Load bag
bag_file = "./data/dataset_path"
bag = bagreader(bag_file)

# Extract raw contact data
csvfile_raw = bag.message_by_topic("/unitree/raw_contact")
df_raw = pd.read_csv(csvfile_raw)

# Extract contact state data
csvfile_state = bag.message_by_topic("/unitree/contact_state")
df_state = pd.read_csv(csvfile_state)

print("\nContact state columns:", df_state.columns.tolist())

# Contact columns for raw data
#contact_cols = ['raw_contacts_0', 'raw_contacts_1', 'raw_contacts_2', 'raw_contacts_3']
contact_cols = ['ee_state_0', 'ee_state_1', 'ee_state_2', 'ee_state_3']

# Contact state columns (assuming they follow similar naming convention)
# You might need to adjust these column names based on your actual data structure

#state_cols = ['contact_state_0', 'contact_state_1', 'contact_state_2', 'contact_state_3']
state_cols = ['ee_state_0', 'ee_state_1', 'ee_state_2', 'ee_state_3']

# Create subplots (4 rows, 1 column)
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# Colors for each contact
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

for i, (raw_col, state_col) in enumerate(zip(contact_cols, state_cols)):
    # Plot raw contact signal
    axes[i].plot(df_raw["Time"], df_raw[raw_col], color=colors[i], alpha=0.7, linewidth=1, label='Raw Signal')

    # Create twin axis for boolean overlay
    ax2 = axes[i].twinx()

    # Plot boolean contact state as filled areas
    if state_col in df_state.columns:
        # Convert boolean to numeric for plotting
        contact_bool = df_state[state_col].astype(int)

        # Get y-limits of the raw signal for scaling the boolean overlay
        y_min, y_max = axes[i].get_ylim()
        y_range = y_max - y_min

        # Scale boolean signal to be visible but not overwhelming
        scaled_bool = contact_bool * y_range * 0.1 + y_min

        # Fill areas where contact is True
        ax2.fill_between(df_state["Time"], 0, contact_bool,
                        alpha=0.3, color='black', label='Contact State')

        # Also plot as line for clarity
        ax2.plot(df_state["Time"], contact_bool,
                color='black', linewidth=2, alpha=0.8)

        ax2.set_ylim(0, 1.2)  # Give some space above the boolean signal
        ax2.set_ylabel('Contact State', fontsize=10)
        ax2.tick_params(axis='y', labelsize=8)
    else:
        print(f"Warning: Column {state_col} not found in contact state data")

    # Format the main axis
    axes[i].set_ylabel("Raw Value [a.u.]", fontsize=10)
    axes[i].set_title(f"{raw_col} Contact Signal with Boolean State Overlay", fontsize=12)
    axes[i].grid(True, alpha=0.3)
    axes[i].tick_params(axis='y', labelsize=8)

    # Add legends
    lines1, labels1 = axes[i].get_legend_handles_labels()
    if state_col in df_state.columns:
        lines2, labels2 = ax2.get_legend_handles_labels()
        axes[i].legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)
    else:
        axes[i].legend(loc='upper right', fontsize=8)

# Label the shared x-axis
axes[-1].set_xlabel("Time [s]", fontsize=12)
plt.tight_layout()
plt.show()

# Print some info about the data
print("Raw contact data shape:", df_raw.shape)
print("Contact state data shape:", df_state.shape)
print("\nRaw contact columns:", df_raw.columns.tolist())
print("\nContact state columns:", df_state.columns.tolist())

# Show timing comparison
print(f"\nRaw data time range: {df_raw['Time'].min():.2f} to {df_raw['Time'].max():.2f} seconds")
print(f"State data time range: {df_state['Time'].min():.2f} to {df_state['Time'].max():.2f} seconds")

In [ ]:
#----------------------------------HYSTERISIS PLOTTING AND THRESHOLDING--------------------------------------
bag_file = "./data/dataset_path"
import bagpy
from bagpy import bagreader
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

# Load the data (assuming you already have this)
#bag_file = "./data/dataset_path"
bag = bagreader(bag_file)
csvfile_raw = bag.message_by_topic("/unitree/raw_contact")
df_raw = pd.read_csv(csvfile_raw)

# Contact columns
contact_cols = ['raw_contacts_0', 'raw_contacts_1', 'raw_contacts_2', 'raw_contacts_3']
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

# Function to detect optimal threshold using multiple methods
def analyze_contact_thresholds(data, col_name):
    """Analyze data to suggest optimal contact detection threshold
       So we do a thorough analysing of the data here"""

    # Method 1: Statistical analysis
    mean_val = np.mean(data)
    std_val = np.std(data)
    median_val = np.median(data)

    # Method 2: Histogram-based (find valley between peaks if bimodal)
    hist, bin_edges = np.histogram(data, bins=50)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    # Method 3: Otsu's method (adapted for threshold selection)
    def otsu_threshold(data):
        hist, bin_edges = np.histogram(data, bins=256)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

        # Normalize histogram
        hist = hist.astype(float) / hist.sum()

        max_variance = 0
        best_threshold = 0

        for i in range(1, len(hist)-1):
            # Background and foreground weights
            w0 = hist[:i].sum()
            w1 = hist[i:].sum()

            if w0 == 0 or w1 == 0:
                continue

            # Background and foreground means
            mu0 = (hist[:i] * bin_centers[:i]).sum() / w0 if w0 > 0 else 0
            mu1 = (hist[i:] * bin_centers[i:]).sum() / w1 if w1 > 0 else 0

            # Between-class variance
            variance = w0 * w1 * (mu0 - mu1)**2

            if variance > max_variance:
                max_variance = variance
                best_threshold = bin_centers[i]

        return best_threshold

    otsu_thresh = otsu_threshold(data)

    print(f"\n=== Threshold Analysis for {col_name} ===")
    print(f"Mean: {mean_val:.1f}")
    print(f"Median: {median_val:.1f}")
    print(f"Std: {std_val:.1f}")
    print(f"Mean + 0.5*Std: {mean_val + 0.5*std_val:.1f}")
    print(f"Mean + 1*Std: {mean_val + std_val:.1f}")
    print(f"Otsu threshold: {otsu_thresh:.1f}")
    print(f"Data range: {np.min(data):.1f} to {np.max(data):.1f}")

    return {
        'mean': mean_val,
        'median': median_val,
        'std': std_val,
        'mean_plus_half_std': mean_val + 0.5*std_val,
        'mean_plus_std': mean_val + std_val,
        'otsu': otsu_thresh
    }

# Analyze each sensor
threshold_analysis = {}
for col in contact_cols:
    threshold_analysis[col] = analyze_contact_thresholds(df_raw[col], col)

# Manual threshold setting based on analysis
# You can adjust these based on the analysis output
default_thresholds = {
    'raw_contacts_0': 300 - 100,  # Adjust based on analysis
    'raw_contacts_1': 500 - 150,  # Adjust based on analysis
    'raw_contacts_2': 100 - 50,  # Adjust based on analysis
    'raw_contacts_3': 450 - 100,  # Adjust based on analysis (higher due to sensitivity)
}


# Function to apply threshold-based contact detection
def detect_contacts_threshold(data, threshold, hysteresis_factor=0.9, min_contact_duration=0.01):
    """
    Apply threshold-based contact detection with hysteresis and minimum duration filtering

    Parameters:
    - data: raw sensor data
    - threshold: contact detection threshold
    - hysteresis_factor: factor for lower threshold (prevents chattering)
    - Common “rule of thumb”  hysterisis of 5–15% below the main threshold usually works well for sensors that are moderately noisy.
    - min_contact_duration: minimum contact duration in seconds (1000Hz sampling)
    """

    # Hysteresis thresholds
    upper_thresh = threshold
    lower_thresh = threshold * hysteresis_factor

    # Initialize contact state
    contacts = np.zeros(len(data), dtype=bool)
    current_state = False

    # Apply hysteresis
    for i in range(len(data)):
        if not current_state and data[i] > upper_thresh:
            current_state = True
        elif current_state and data[i] < lower_thresh:
            current_state = False
        contacts[i] = current_state

    # Apply minimum duration filtering (remove very short contacts/gaps)
    # At500Hz sampling rate
    min_samples = int(min_contact_duration * 500)

    # Remove short contact periods
    contact_diff = np.diff(contacts.astype(int))
    contact_starts = np.where(contact_diff == 1)[0] + 1
    contact_ends = np.where(contact_diff == -1)[0] + 1

    # Handle edge cases
    if contacts[0]:
        contact_starts = np.insert(contact_starts, 0, 0)
    if contacts[-1]:
        contact_ends = np.append(contact_ends, len(contacts))


    # Filter short contacts
    ls,le = None,None
    for start, end in zip(contact_starts, contact_ends):

        if ls and le:
            if start - le < min_samples:
                contacts[le:start] = True
            pass

        if end - start < min_samples:
            ls,le = start,end
            #contacts[start:end] = False

    return contacts

# Apply threshold-based detection
print("Applying threshold-based contact detection...")
detected_contacts = {}

for col in contact_cols:
    # Use suggested threshold or manual override
    threshold = default_thresholds[col]

    # Detect contacts
    detected_contacts[col] = detect_contacts_threshold(
        df_raw[col],
        threshold=threshold,
        hysteresis_factor=0.9,
        min_contact_duration=0.05  # 50ms minimum contact
    )

    # Calculate contact statistics
    contact_percentage = np.mean(detected_contacts[col]) * 100
    print(f"{col}: Threshold={threshold:.1f}, Contact={contact_percentage:.1f}% of time")

# Create visualization comparing original vs threshold-based detection
fig, axes = plt.subplots(4, 1, figsize=(16*2, 12*2), sharex=True)

for i, col in enumerate(contact_cols):

    # Plot raw signal
    axes[i].plot(df_raw["Time"], df_raw[col], color=colors[i], alpha=0.7, linewidth=1, label='Raw Signal')

    # Plot threshold line
    axes[i].axhline(y=default_thresholds[col], color='red', linestyle='--', alpha=0.8, label=f'Threshold ({default_thresholds[col]})')

    axes[i].axhline(y=default_thresholds[col]*0.9, color='red', linestyle='--', alpha=0.8, label=f'Threshold ({default_thresholds[col]})')

    # Create twin axis for contact states
    ax2 = axes[i].twinx()


    # Plot threshold-based detection
    #ax2.plot(df_raw["Time"], detected_contacts[col].astype(int),
    #        color='darkgreen', linewidth=2, alpha=0.8)

    ax2.fill_between(df_raw["Time"], 0, detected_contacts[col].astype(int),
                    alpha=0.2, color='green', label='Threshold-Based Contact')

    ax2.set_ylim(-0.1, 1.3)
    ax2.set_ylabel('Contact State', fontsize=10)
    ax2.tick_params(axis='y', labelsize=8)

    # Format main axis
    axes[i].set_ylabel("Raw Value [a.u.]", fontsize=10)
    axes[i].set_title(f"{col} - Threshold-Based Contact Detection", fontsize=12)
    axes[i].grid(True, alpha=0.3)
    axes[i].tick_params(axis='y', labelsize=8)

    # Combine legends
    lines1, labels1 = axes[i].get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    axes[i].legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)

axes[-1].set_xlabel("Time [s]", fontsize=12)
plt.tight_layout()
plt.show()

# Save the improved contact states
contact_df = df_raw[['Time'] + contact_cols].copy()
for col in contact_cols:
    contact_df[f'{col}_contact_threshold'] = detected_contacts[col].astype(int)

print(f"\nContact detection complete!")
print(f"New dataframe shape: {contact_df.shape}")
print(f"Columns: {contact_df.columns.tolist()}")

# Export for ML training
contact_df.to_csv('concrete_improved_contact_labels.csv', index=False)
print("Saved improved contact labels to 'concrete_improved_contact_labels.csv'")

    # Display sample of the results
print("\nSample of improved contact labels:")
print(contact_df[['Time', 'raw_contacts_0', 'raw_contacts_0_contact_threshold',
                 'raw_contacts_3', 'raw_contacts_3_contact_threshold']].head(10))

In [ ]:
import os
import zipfile
import tempfile
import shutil
from bagpy import bagreader
import pandas as pd
import numpy as np
import re
from scipy.spatial.transform import Rotation as R

def open_zipped_bag(zip_path):
    """Temporarily extract a .bag from a .zip and return its path + temp dir."""
    temp_dir = tempfile.mkdtemp()
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(temp_dir)
    for file in os.listdir(temp_dir):
        if file.endswith('.bag'):
            return os.path.join(temp_dir, file), temp_dir
    raise FileNotFoundError(f"No .bag file found inside {zip_path}")        

        
    # find the first .bag inside
    for file in os.listdir(temp_dir):
        if file.endswith('.bag'):
            return os.path.join(temp_dir, file), temp_dir
    raise FileNotFoundError("No .bag file inside zip!")

def extract_values(string_data):
    pattern = r'[-+]?\d*\.\d+(?:[eE][-+]?\d+)?|[-+]?\d+[eE][-+]?\d+|[-+]?\d+'
    numbers = re.findall(pattern, string_data)
    return np.array([float(num) for num in numbers], dtype=float)

bag_paths = [
    #"./data/dataset_path",
    #"./data/dataset_path",
    #"./data/dataset_path",
    #"./data/dataset_path",
    #"./data/dataset_path",
    #"./data/dataset_path",
    #"./data/dataset_path",
    #"./data/dataset_path",
    #"./data/dataset_path",
    #"./data/dataset_path",
    #"./data/dataset_path"
    #"./data/dataset_path"
    #"./data/dataset_path"
    #"./data/dataset_path"
    "./data/dataset_path"
  
]

output_csv_dir = "./data/dataset_path"
os.makedirs(output_csv_dir, exist_ok=True)

#Loop through each bag
for bag_path in bag_paths:
    print(f"\n Processing {bag_path}...")
    
    temp_dir = None
    try:
        # Handle zipped .bag files
        if bag_path.endswith(".zip"):
            bag_path, temp_dir = open_zipped_bag(bag_path)

        b = bagreader(bag_path)
        state_csv_path = b.message_by_topic("/estimated_robot_state")
        imu_csv = b.message_by_topic("/unitree/imu")

        state_df = pd.read_csv(state_csv_path)
        imu_df = pd.read_csv(imu_csv)

        # Extract values from columns
        state_df['ee_pos'] = state_df['ee_pos'].apply(lambda x: extract_values(str(x)))
        state_df['ee_vel'] = state_df['ee_vel'].apply(lambda x: extract_values(str(x)))
        state_df['ee_forces_cleaned'] = state_df['ee_forces'].apply(lambda x: extract_values(str(x)))


        # Extract Linear Acceleration &  Angular VelocityunitreeRawContactMsg_
        acc = imu_df[["linear_acceleration.x", "linear_acceleration.y", "linear_acceleration.z"]].to_numpy()
        omega = imu_df[["angular_velocity.x", "angular_velocity.y", "angular_velocity.z"]].to_numpy()
        # Save CSV with unique name
        # Save CSVs
        base_name = os.path.splitext(os.path.basename(bag_path))[0]
        output_csv_path = os.path.join(output_csv_dir, f"{base_name}_cleaned.csv")
        state_df.to_csv(output_csv_path, index=False)
        print(f"Saved cleaned CSV: {output_csv_path}")

        imu_output_csv_path = os.path.join(output_csv_dir, f"{base_name}_imu.csv")
        imu_df.to_csv(imu_output_csv_path, index=False)
        print(f"Saved IMU CSV: {imu_output_csv_path}")
        
    except Exception as e:
        print(f"Failed to process {bag_path} because: {e}")
        
    finally:
        if temp_dir:
            shutil.rmtree(temp_dir)

In [ ]:
#!/usr/bin/env python3
import os, re
import numpy as np
import pandas as pd
from bagpy import bagreader
from scipy.spatial.transform import Rotation as R
import matplotlib.pyplot as plt

# ============================================================
# ------------------- UTILITY FUNCTIONS ----------------------
# ============================================================

def extract_values(string_data):
    """Extract numeric values from string using regex."""
    pattern = r'[-+]?\d*\.\d+(?:[eE][-+]?\d+)?|[-+]?\d+[eE][-+]?\d+|[-+]?\d+'
    numbers = re.findall(pattern, str(string_data))
    return np.array([float(num) for num in numbers], dtype=float)

def is_in_body_frame(ee_array):
    """Quick heuristic: if all magnitudes < 5 m → body frame."""
    flat = ee_array.flatten()
    return np.all(np.abs(flat) < 5.0)

def parse_vec12(series):
    """Parse each row of 12 numeric values (flattened [LF RF LH RH])."""
    n = len(series)
    out = np.full((n, 12), np.nan, dtype=float)
    for i, s in enumerate(series.values):
        v = np.fromstring(str(s).strip().strip('[]').replace(',', ' '), sep=' ')
        if v.size >= 12:
            out[i, :12] = v[:12]
    return out

def rot_mats(quat_xyzw):
    """Compute rotation matrices Rwb and Rbw from quaternions."""
    q = quat_xyzw.copy()
    n = np.linalg.norm(q, axis=1, keepdims=True)
    n[n == 0] = 1.0
    Rwb = np.array([R.from_quat(qi / ni).as_matrix() for qi, ni in zip(q, n.squeeze())])
    Rbw = np.transpose(Rwb, (0, 2, 1))
    return Rwb, Rbw

def rt_err_pos(p12, base_pos, Rwb, Rbw, idx):
    """Compute round-trip positional error."""
    p = p12[idx].reshape(-1, 4, 3)
    bp = base_pos[idx][:, None, :]
    pb = np.einsum('nij,nkj->nki', Rbw[idx], p - bp)
    pw = np.einsum('nij,nkj->nki', Rwb[idx], pb) + bp
    eW = np.linalg.norm(p - pw, axis=2).mean()
    pw2 = np.einsum('nij,nkj->nki', Rwb[idx], p) + bp
    eB = np.linalg.norm(p - pw2, axis=2).mean()
    return eW, eB

def rt_err_vel(v12, p12W, base_pos, v_base, w_base, Rwb, Rbw, idx):
    """Compute round-trip velocity error."""
    v  = v12[idx].reshape(-1, 4, 3)
    pW = p12W[idx].reshape(-1, 4, 3)
    bp = base_pos[idx][:, None, :]
    vb = v_base[idx][:, None, :]
    wb = w_base[idx][:, None, :]
    rW  = pW - bp
    wxr = np.cross(wb, rW)
    vb_assume = np.einsum('nij,nkj->nki', Rbw[idx], v - vb - wxr)
    vw_back   = np.einsum('nij,nkj->nki', Rwb[idx], vb_assume) + vb + wxr
    eW = np.linalg.norm(v - vw_back, axis=2).mean()
    vw_assume = np.einsum('nij,nkj->nki', Rwb[idx], v) + vb + wxr
    eB = np.linalg.norm(v - vw_assume, axis=2).mean()
    return eW, eB

def detect(eW, eB, tol=1e-6):
    """Classify frame based on round-trip errors."""
    if eW + tol < eB: return "WORLD"
    if eB + tol < eW: return "BODY"
    return "AMBIG"

def plot_foot_z(state_df, base_name):
    """Plot z-position of each leg over time."""
    ee_pos_array = np.vstack(state_df['ee_pos_cleaned'].to_numpy())
    timesteps = np.arange(len(state_df))
    leg_labels = ["LF", "RF", "LH", "RH"]
    colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

    fig, ax = plt.subplots(figsize=(10, 6))
    for i, leg in enumerate(leg_labels):
        z_vals = ee_pos_array[:, i*3 + 2]
        ax.plot(timesteps, z_vals, label=f"{leg} z", color=colors[i])
    ax.set_title(f"Foot Z-Position Over Time ({base_name})")
    ax.set_xlabel("Timestep")
    ax.set_ylabel("Z Position (m)")
    ax.legend()
    ax.grid(True)
    plt.tight_layout()
    plt.show()

# ============================================================
# -------------------- MAIN PROCESS --------------------------
# ============================================================

bag_paths = [
    "./data/dataset_path"
]

output_csv_dir = "./data/dataset_path"
os.makedirs(output_csv_dir, exist_ok=True)
LEGS = ["LF", "RF", "LH", "RH"]

for bag_path in bag_paths:
    print(f"\nProcessing {bag_path}...")
    try:
        b = bagreader(bag_path)
        state_csv_path = b.message_by_topic("/estimated_robot_state")
        imu_csv_path   = b.message_by_topic("/unitree/imu")
        print(f"State CSV: {state_csv_path}")

        state_df = pd.read_csv(state_csv_path)
        imu_df   = pd.read_csv(imu_csv_path)

        # ---- Extract vectors
        state_df['ee_pos'] = state_df['ee_pos'].apply(extract_values)
        state_df['ee_vel'] = state_df['ee_vel'].apply(extract_values)
        state_df['ee_forces_cleaned'] = state_df['ee_forces'].apply(extract_values)

        # ---- Detect frame
        first_ee = np.vstack(state_df['ee_pos'].to_numpy())
        if np.max(np.abs(first_ee)) > 5.0:
            print("Detected WORLD frame → converting to BODY frame...")
        else:
            print("Detected BODY frame → using as-is.")

        if is_in_body_frame(first_ee):
            print("Already in BODY frame.")
            state_df['ee_pos_cleaned'] = state_df['ee_pos']
            state_df['ee_vel_cleaned'] = state_df['ee_vel']
        else:
            print("Converting WORLD → BODY frame...")
            base_pos  = state_df[['base_pose.position.x', 'base_pose.position.y', 'base_pose.position.z']].to_numpy()
            base_quat = state_df[['base_pose.orientation.x','base_pose.orientation.y','base_pose.orientation.z','base_pose.orientation.w']].to_numpy()
            ee_pos_world = np.vstack(state_df['ee_pos'].to_numpy())
            ee_vel_world = np.vstack(state_df['ee_vel'].to_numpy())

            r_all = R.from_quat(base_quat)
            ee_pos_body = np.zeros_like(ee_pos_world)
            ee_vel_body = np.zeros_like(ee_vel_world)

            for i in range(4):
                pos_diff = ee_pos_world[:, i*3:(i+1)*3] - base_pos
                ee_pos_body[:, i*3:(i+1)*3] = r_all.inv().apply(pos_diff)
                ee_vel_body[:, i*3:(i+1)*3] = r_all.inv().apply(ee_vel_world[:, i*3:(i+1)*3])

            state_df['ee_pos_cleaned'] = list(ee_pos_body)
            state_df['ee_vel_cleaned'] = list(ee_vel_body)

        # ---- Round-trip consistency test
        pos12 = parse_vec12(state_df['ee_pos_cleaned'])
        vel12 = parse_vec12(state_df['ee_vel_cleaned'])
        base_pos  = state_df[['base_pose.position.x','base_pose.position.y','base_pose.position.z']].to_numpy()
        base_quat = state_df[['base_pose.orientation.x','base_pose.orientation.y','base_pose.orientation.z','base_pose.orientation.w']].to_numpy()
        v_base    = state_df[['base_vel.linear.x','base_vel.linear.y','base_vel.linear.z']].to_numpy()
        w_base    = state_df[['base_vel.angular.x','base_vel.angular.y','base_vel.angular.z']].to_numpy()
        Rwb, Rbw  = rot_mats(base_quat)

        N = min(len(pos12), len(vel12), len(state_df))
        pos12, vel12 = pos12[:N], vel12[:N]
        base_pos, v_base, w_base, Rwb, Rbw = base_pos[:N], v_base[:N], w_base[:N], Rwb[:N], Rbw[:N]
        idx = np.linspace(0, N-1, num=min(N, 4000), dtype=int)

        eWp, eBp = rt_err_pos(pos12, base_pos, Rwb, Rbw, idx)
        frame_p  = detect(eWp, eBp)

        if frame_p == "BODY":
            pW_for_v = (np.einsum('nij,nkj->nki', Rwb, pos12.reshape(N,4,3)) + base_pos[:,None,:]).reshape(N,12)
        else:
            pW_for_v = pos12

        eWv, eBv = rt_err_vel(vel12, pW_for_v, base_pos, v_base, w_base, Rwb, Rbw, idx)
        frame_v = detect(eWv, eBv)

        # ---- Convert to BODY frame if needed
        if frame_p == "WORLD":
            pB = np.einsum('nij,nkj->nki', Rbw, pos12.reshape(N,4,3) - base_pos[:,None,:])
        else:
            pB = pos12.reshape(N,4,3)

        if frame_v == "WORLD":
            vW = vel12.reshape(N,4,3)
            if frame_p == "BODY":
                pW = np.einsum('nij,nkj->nki', Rwb, pB) + base_pos[:,None,:]
            else:
                pW = vel12.reshape(N,4,3)
            rW  = pW - base_pos[:,None,:]
            wxr = np.cross(w_base[:,None,:], rW)
            v_rel_w = vW - v_base[:,None,:] - wxr
            vB = np.einsum('nij,nkj->nki', Rbw, v_rel_w)
        else:
            vB = vel12.reshape(N,4,3)

        # ---- Save processed CSV
        base_name = os.path.splitext(os.path.basename(bag_path))[0]
        output_csv_path = os.path.join(output_csv_dir, f"{base_name}_cleaned.csv")
        out = state_df.copy()

        for i, leg in enumerate(LEGS):
            out[f'ee_pos.{leg}.body.x'] = pB[:,i,0]
            out[f'ee_pos.{leg}.body.y'] = pB[:,i,1]
            out[f'ee_pos.{leg}.body.z'] = pB[:,i,2]
            out[f'ee_vel.{leg}.body.x'] = vB[:,i,0]
            out[f'ee_vel.{leg}.body.y'] = vB[:,i,1]
            out[f'ee_vel.{leg}.body.z'] = vB[:,i,2]

        out['__ee_pos_frame__'] = frame_p
        out['__ee_vel_frame__'] = frame_v
        out.to_csv(output_csv_path, index=False)
        print(f"{base_name} → pos:{frame_p} | vel:{frame_v}")
        print(f"Saved: {output_csv_path}")

        imu_output_csv_path = os.path.join(output_csv_dir, f"{base_name}_imu.csv")
        imu_df.to_csv(imu_output_csv_path, index=False)
        print(f"Saved IMU CSV: {imu_output_csv_path}")

        # ---- Plot foot & base positions
        plot_foot_z(state_df, base_name)
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(np.arange(len(state_df)), state_df['base_pose.position.z'], label="Base Z-Position")
        ax.set_title(f"Base Z-Position Over Time ({base_name})")
        ax.set_xlabel("Timestep")
        ax.set_ylabel("Z Position (m)")
        ax.legend(); ax.grid(True)
        plt.tight_layout(); plt.show()

    except Exception as e:
        print(f"Failed to process {bag_path}: {e}")


In [ ]:
import pandas as pd

# Load files
cleaned_df = pd.read_csv("./data/dataset_path")
thresholded_df = pd.read_csv("concrete_improved_contact_labels.csv")

print("Length cleaned:", len(cleaned_df))
print("Length thresholded:", len(thresholded_df))

# Ensure Time columns are sorted (REQUIRED for merge_asof)
cleaned_df = cleaned_df.sort_values("Time").reset_index(drop=True)
thresholded_df = thresholded_df.sort_values("Time").reset_index(drop=True)

# ---- NEAREST-TIMESTAMP MERGE ----
merged = pd.merge_asof(
    cleaned_df,
    thresholded_df[
        [
            "Time",
            "raw_contacts_0_contact_threshold",
            "raw_contacts_1_contact_threshold",
            "raw_contacts_2_contact_threshold",
            "raw_contacts_3_contact_threshold",
        ]
    ],
    on="Time",
    direction="nearest",
    tolerance=0.002,      # 2 milliseconds tolerance
)

# Remove rows where no match was found
merged = merged.dropna(subset=["raw_contacts_0_contact_threshold"])

print("Length merged after nearest:", len(merged))

# ---- REBUILD ee_contact LIST-COLUMN ----
merged["ee_contact"] = merged[
    [
        "raw_contacts_0_contact_threshold",
        "raw_contacts_1_contact_threshold",
        "raw_contacts_2_contact_threshold",
        "raw_contacts_3_contact_threshold",
    ]
].astype(int).values.tolist()

# ---- SAVE ----
output_path = "./data/dataset_path"
merged.to_csv(output_path, index=False)



print("Saved:", output_path)


In [ ]:
import numpy as np
import pandas as pd

# ===== LOAD MERGED CONTACT LABELS (REFINED CONTACTS) =====
contact_labels_path = "./data/dataset_path"
contact_df = pd.read_csv(contact_labels_path)

print("\nLoaded refined contact labels from:", contact_labels_path)
print("First 5 refined contact rows:")
print(contact_df[[
    "raw_contacts_0_contact_threshold",
    "raw_contacts_1_contact_threshold",
    "raw_contacts_2_contact_threshold",
    "raw_contacts_3_contact_threshold"
]].head())


In [ ]:
#!/usr/bin/env python3
import os
import re
import ast
import numpy as np
import pandas as pd
#-------------------------
# ok what does this file here do?
# This file is for loading the cleaned CSVs, aligning them with the refined contact labels (which have been improved using thresholding)
# , and preparing the data for ML training. In case of the new files i do not need improved thresholding so we have to comment that out 
# and just load the cleaned CSVs and align them with the refined contact labels. The main goal is to ensure that the ee_contact column 
# in the cleaned CSVs is replaced with the improved contact labels from the refined CSV, and then save the merged result for ML training.
# i also dont need improved contacts so....
# -------------------------
# Helpers
# -------------------------
def fix_bracket_list_str(s):
    if not isinstance(s, str):
        return s
    s = s.strip()
    if s.startswith('[') and s.endswith(']'):
        content = s[1:-1].strip()
        parts = re.split(r'\s+', content)
        return "[" + ",".join(parts) + "]"
    return s

def safe_literal_eval(val):
    if pd.isna(val):
        return None
    try:
        val_fixed = fix_bracket_list_str(val)
        return ast.literal_eval(val_fixed)
    except Exception:
        try:
            nums = re.findall(r'[-+]?\d*\.\d+(?:[eE][-+]?\d+)?|[-+]?\d+', str(val))
            return [float(x) for x in nums]
        except Exception:
            return None

def to_fixed_length_array(x, length=12):
    if x is None:
        return np.full(length, np.nan, dtype=float)
    if isinstance(x, np.ndarray):
        arr = x
    else:
        try:
            arr = np.array(x, dtype=float)
        except Exception:
            return np.full(length, np.nan, dtype=float)
    if arr.size == length:
        return arr.astype(float)
    if arr.size < length:
        padded = np.full(length, np.nan, dtype=float)
        padded[:arr.size] = arr
        return padded
    return arr[:length].astype(float)

def validate_and_stack(name, arr, expected_length=12):
    print(f"\nValidating: {name}")
    valid = []
    for i, item in enumerate(arr):
        if isinstance(item, np.ndarray) and item.shape[0] == expected_length and np.issubdtype(item.dtype, np.floating):
            if not np.all(np.isnan(item)):
                valid.append(item)
            else:
                print(f"{name}[{i}] all-NaN, skipping")
        else:
            length = item.shape[0] if isinstance(item, np.ndarray) else (len(item) if hasattr(item, "__len__") else "N/A")
            print(f"{name}[{i}] invalid. Type: {type(item)}, Length: {length}")
    if not valid:
        print(f"No valid entries in {name}")
        return np.empty((0, expected_length))
    res = np.vstack(valid)
    print(f"{name} shape: {res.shape}")
    return res

def quat_xyzw_to_rotmat(x, y, z, w):
    """
    Quaternion given as (x, y, z, w) -> rotation matrix 3x3
    Follows same formula used earlier: q = [x,y,z,w].
    """
    xx = x * x
    yy = y * y
    zz = z * z
    ww = w * w
    xy = x * y
    xz = x * z
    xw = x * w
    yz = y * z
    yw = y * w
    zw = z * w

    R = np.array([
        [1 - 2*(yy + zz),     2*(xy - xw*0 - zw),    2*(xz + yw)], 
        [2*(xy + zw),         1 - 2*(xx + zz),       2*(yz - xw)],
        [2*(xz - yw),         2*(yz + xw),           1 - 2*(xx + yy)]
    ])
    # The above uses shorter expressions; ensure correctness by using standard formula:
    R = np.array([
        [1 - 2*(yy + zz), 2*(xy - w*z),     2*(xz + w*y)],
        [2*(xy + w*z),    1 - 2*(xx + zz),  2*(yz - w*x)],
        [2*(xz - w*y),    2*(yz + w*x),     1 - 2*(xx + yy)]
    ])
    return R

# -------------------------
# Configuration
# -------------------------
keyword = "legro_2025-06-09-14-23-57"
cleaned_csv_dir = "./data/dataset_path"
refined_contact_path = "./data/dataset_path"
output_dir = "./data/dataset_path"
os.makedirs(output_dir, exist_ok=True)

contact_cols = [
    "raw_contacts_0_contact_threshold",
    "raw_contacts_1_contact_threshold",
    "raw_contacts_2_contact_threshold",
    "raw_contacts_3_contact_threshold",
]

# -------------------------
# Load refined contacts once (must contain Time)
# -------------------------
print("Loading refined contact labels from:", refined_contact_path)
refined_df = pd.read_csv(refined_contact_path)
print("Refined contact columns:", refined_df.columns.tolist())
if "Time" not in refined_df.columns:
    raise RuntimeError("Refined contact CSV must contain a 'Time' column for alignment.")
refined_df["Time"] = refined_df["Time"].astype(float)
refined_df = refined_df.sort_values("Time").reset_index(drop=True)
print("First 5 refined contact rows:")
print(refined_df[["Time"] + contact_cols].head())

# -------------------------
# Process each cleaned CSV
# -------------------------
cleaned_files = sorted([
    os.path.join(cleaned_csv_dir, f)
    for f in os.listdir(cleaned_csv_dir)
    if f.endswith("_cleaned.csv")
])
print(f"\nFound {len(cleaned_files)} cleaned CSV files.")

for file in cleaned_files:
    if keyword not in file:
        continue

    try:
        base_name = os.path.basename(file).replace("_cleaned.csv", "")
        print(f"\nProcessing: {base_name}")

        # Load cleaned state and imu (if present)
        state_df = pd.read_csv(file)
        imu_file = file.replace("_cleaned.csv", "_imu.csv")
        imu_df = pd.read_csv(imu_file) if os.path.exists(imu_file) else pd.DataFrame()

        if "Time" not in state_df.columns:
            raise RuntimeError(f"State CSV {file} does not contain 'Time' column.")
        state_df["Time"] = state_df["Time"].astype(float)
        state_df = state_df.sort_values("Time").reset_index(drop=True)

        # Align refined contacts to this state_df using nearest timestamp (merge_asof)
        merged = pd.merge_asof(
            state_df[["Time"]],
            refined_df[["Time"] + contact_cols],
            on="Time",
            direction="nearest",
            tolerance=0.01
        )

        mask_matched = merged[contact_cols].notnull().all(axis=1)
        if mask_matched.sum() == 0:
            print("No matched rows between state_df and refined contacts within tolerance; skipping file.")
            continue
        if (~mask_matched).any():
            print(f"Warning: {(~mask_matched).sum()} rows had no nearby contact match and will be skipped.")

        matched_state_df = state_df[mask_matched].reset_index(drop=True)
        aligned_contacts = merged.loc[mask_matched, contact_cols].to_numpy().astype(int)
        print("Aligned contacts shape:", aligned_contacts.shape)
        print("Matched state rows:", len(matched_state_df))

        # -------------------------
        # Parse EE pos / vel / forces from matched_state_df
        # -------------------------
        use_pos_col = "ee_pos_cleaned" if "ee_pos_cleaned" in matched_state_df.columns else "ee_pos"
        use_vel_col = "ee_vel_cleaned" if "ee_vel_cleaned" in matched_state_df.columns else "ee_vel"
        use_forces_col = "ee_forces_cleaned" if "ee_forces_cleaned" in matched_state_df.columns else "ee_forces"

        print("Using position column:", use_pos_col)
        print("Using velocity column:", use_vel_col)
        print("Using forces column:", use_forces_col)

        parsed_pos = matched_state_df[use_pos_col].apply(safe_literal_eval).apply(lambda x: to_fixed_length_array(x, 12)).to_list()
        ee_pos_array = np.vstack(parsed_pos) if parsed_pos else np.empty((0,12))

        parsed_vel = matched_state_df[use_vel_col].apply(safe_literal_eval).apply(lambda x: to_fixed_length_array(x, 12)).to_list()
        ee_vel_array = np.vstack(parsed_vel) if parsed_vel else np.empty((0,12))

        parsed_forces = matched_state_df[use_forces_col].apply(safe_literal_eval).apply(lambda x: to_fixed_length_array(x, 12)).to_list()
        ee_forces_array = np.vstack(parsed_forces) if parsed_forces else np.empty((0,12))

        # IMU arrays
        acc = imu_df[["linear_acceleration.x", "linear_acceleration.y", "linear_acceleration.z"]].to_numpy() if not imu_df.empty else np.empty((0, 3))
        omega = imu_df[["angular_velocity.x", "angular_velocity.y", "angular_velocity.z"]].to_numpy() if not imu_df.empty else np.empty((0, 3))

        # Parse joint states
        try:
            q = np.array(matched_state_df["joint_state.position"].apply(safe_literal_eval).apply(lambda x: to_fixed_length_array(x, 12)).to_list())
            qd = np.array(matched_state_df["joint_state.velocity"].apply(safe_literal_eval).apply(lambda x: to_fixed_length_array(x, 12)).to_list())
        except Exception as e:
            print("Error parsing joint states:", e)
            q = qd = np.empty((0, 12))

        # Validate (diagnostics)
        p_valid = validate_and_stack("ee_pos", ee_pos_array, expected_length=12)
        v_valid = validate_and_stack("ee_vel", ee_vel_array, expected_length=12)

        # Summary
        print("\nFinal Preprocessed Data Shapes (Before Trimming):")
        print(f"  Aligned Contacts: {aligned_contacts.shape}")
        print(f"  q (positions)   : {q.shape}")
        print(f"  qd (velocities) : {qd.shape}")
        print(f"  p (ee_pos)      : {ee_pos_array.shape}")
        print(f"  v (ee_vel)      : {ee_vel_array.shape}")
        print(f"  ee_forces       : {ee_forces_array.shape}")
        print(f"  acc (IMU accel) : {acc.shape}")
        print(f"  omega (IMU ang) : {omega.shape}")

        # Determine min_length across relevant arrays
        arrays_for_length = [aligned_contacts, q, qd, ee_pos_array, ee_vel_array]
        if acc.size > 0:
            arrays_for_length.append(acc)
        if omega.size > 0:
            arrays_for_length.append(omega)

        min_length = min([arr.shape[0] for arr in arrays_for_length]) if arrays_for_length else 0
        print(f"Trimming all to min_length = {min_length}")

        # Trim in-sync
        aligned_contacts = aligned_contacts[:min_length]
        q = q[:min_length]
        qd = qd[:min_length]
        ee_pos_array = ee_pos_array[:min_length]
        ee_vel_array = ee_vel_array[:min_length]
        ee_forces_array = ee_forces_array[:min_length]
        if acc.size > 0:
            acc = acc[:min_length]
        if omega.size > 0:
            omega = omega[:min_length]

        # -------------------------
        # NOW convert ee_pos and ee_vel WORLD -> BODY using per-row base orientation and base position
        # prefer base_pose.orientation.* (x,y,z,w) then fallback to base_ori.* if available
        # -------------------------
        if all(c in matched_state_df.columns for c in ["base_pose.position.x","base_pose.position.y","base_pose.position.z"]):
            base_pos = matched_state_df[["base_pose.position.x","base_pose.position.y","base_pose.position.z"]].to_numpy()[:min_length]
        else:
            raise RuntimeError("Missing base_pose.position.* columns needed for body-frame conversion.")

        # get quaternion columns (prefer base_pose.orientation.x,y,z,w)
        if all(c in matched_state_df.columns for c in ["base_pose.orientation.x","base_pose.orientation.y","base_pose.orientation.z","base_pose.orientation.w"]):
            qx = matched_state_df["base_pose.orientation.x"].to_numpy()[:min_length]
            qy = matched_state_df["base_pose.orientation.y"].to_numpy()[:min_length]
            qz = matched_state_df["base_pose.orientation.z"].to_numpy()[:min_length]
            qw = matched_state_df["base_pose.orientation.w"].to_numpy()[:min_length]
        elif all(c in matched_state_df.columns for c in ["base_ori.x","base_ori.y","base_ori.z","base_ori.w"]):
            qx = matched_state_df["base_ori.x"].to_numpy()[:min_length]
            qy = matched_state_df["base_ori.y"].to_numpy()[:min_length]
            qz = matched_state_df["base_ori.z"].to_numpy()[:min_length]
            qw = matched_state_df["base_ori.w"].to_numpy()[:min_length]
        else:
            raise RuntimeError("Missing quaternion columns for body-frame conversion (expected base_pose.orientation.* or base_ori.*).")

        # Prepare body arrays
        ee_pos_body = np.zeros_like(ee_pos_array)
        ee_vel_body = np.zeros_like(ee_vel_array)

        # Vectorized-ish loop (per-row)
        for i in range(min_length):
            R = quat_xyzw_to_rotmat(qx[i], qy[i], qz[i], qw[i])  # returns R_wb (world->body) if we use transpose later
            R_bw = R.T  # world->body rotation

            # for each leg
            for leg in range(4):
                idx = slice(leg*3, (leg+1)*3)
                pw = ee_pos_array[i, idx]
                vw = ee_vel_array[i, idx]
                pw_rel = pw - base_pos[i]
                # transform
                pb = R_bw.dot(pw_rel)
                vb = R_bw.dot(vw)
                ee_pos_body[i, idx] = pb
                ee_vel_body[i, idx] = vb

        ee_pos_array = ee_pos_body
        ee_vel_array = ee_vel_body
        print("Converted EE kinematics to BODY frame (ee_pos_array, ee_vel_array).")

        # Confirm shapes after trimming + conversion
        print("\nFinal Shapes After Trimming & Conversion:")
        print(f"  Contacts       : {aligned_contacts.shape}")
        print(f"  q (positions)  : {q.shape}")
        print(f"  qd (velocities): {qd.shape}")
        print(f"  p (ee_pos)     : {ee_pos_array.shape}")
        print(f"  v (ee_vel)     : {ee_vel_array.shape}")
        print(f"  acc (IMU accel): {acc.shape}")
        print(f"  omega (IMU ang): {omega.shape}")

        # Frame check helper (informational)
        def check_frame_reference(name, array):
            if array.shape[0] == 0:
                print(f"{name} is empty.")
                return
            max_val = np.nanmax(np.abs(array))
            if max_val > 5:
                print(f"{name} may be in WORLD frame (max abs value = {max_val:.2f})")
            else:
                print(f"{name} appears to be in BODY frame (max abs value = {max_val:.2f})")

        check_frame_reference("p (ee_pos)", ee_pos_array)
        check_frame_reference("v (ee_vel)", ee_vel_array)
        check_frame_reference("q", q)
        check_frame_reference("qd", qd)
        if acc.size > 0:
            check_frame_reference("acc", acc)
        if omega.size > 0:
            check_frame_reference("omega", omega)

        # -------------------------
        # Contacts -> decimal labels (0-15)
        # -------------------------
        contacts_int = aligned_contacts.astype(int)
        reorder_contacts = contacts_int[:, [2, 0, 3, 1]]  # RF, LF, RH, LH
        decimal_contacts = np.dot(reorder_contacts, [8, 4, 2, 1])

        # Reorder joint states and ee pos/vel if necessary
        joint_order = [6,7,8,0,1,2,9,10,11,3,4,5]
        if q.shape[1] >= 12:
            q = q[:, joint_order]
            qd = qd[:, joint_order]
            if ee_pos_array.shape[1] >= 12:
                ee_pos_array = ee_pos_array[:, joint_order]
            if ee_vel_array.shape[1] >= 12:
                ee_vel_array = ee_vel_array[:, joint_order]
        else:
            print("Warning: q has fewer than 12 columns; skipping joint reorder.")

        # Concatenate feature matrix
        to_concat = [q, qd, ee_pos_array, ee_vel_array]
        if acc.size > 0:
            to_concat.append(acc)
        if omega.size > 0:
            to_concat.append(omega)

        if any(arr.size == 0 for arr in to_concat):
            print("One or more arrays are empty — skipping save for this file.")
            continue

        preprocessed_data = np.concatenate(to_concat, axis=1)

        # Save to terrain folder
        terrain_dir = os.path.join(output_dir, base_name)
        os.makedirs(terrain_dir, exist_ok=True)

        print("Saving to:", terrain_dir)
        np.save(os.path.join(terrain_dir, "q.npy"), q)
        np.save(os.path.join(terrain_dir, "qd.npy"), qd)
        np.save(os.path.join(terrain_dir, "p.npy"), ee_pos_array)
        np.save(os.path.join(terrain_dir, "v.npy"), ee_vel_array)
        np.save(os.path.join(terrain_dir, "acc.npy"), acc)
        np.save(os.path.join(terrain_dir, "omega.npy"), omega)
        np.save(os.path.join(terrain_dir, "contacts.npy"), contacts_int)
        np.save(os.path.join(terrain_dir, "contacts_labels.npy"), decimal_contacts)
        np.save(os.path.join(terrain_dir, "preprocessed_data.npy"), preprocessed_data)

        print(f"Saved all data to {terrain_dir}")

    except Exception as e:
        print(f"Failed to process {file} because: {e}")


In [ ]:
import bagpy
from bagpy import bagreader
import pandas as pd
import matplotlib.pyplot as plt

def analyze_contact_data(bag_file):
    """
    Convert ROS bag contact topics to CSV and read the data.
    """

    # Initialize bag reader
    bag = bagreader(bag_file)

    print("Converting topics to CSV...")

    # Convert raw contact data to CSV
    try:
        raw_contact_csv = bag.message_by_topic('/unitree/contact_state')
        print(f"✓ Contact state data converted: {raw_contact_csv}")
    except Exception as e:
        print(f"✗ Failed to convert /unitree/contact_state: {e}")
        return None

    # Read the CSV file
    print("\nReading CSV file...")
    raw_contact_df = pd.read_csv(raw_contact_csv)
    #save raw_contact_csv
    raw_contact_df.to_csv("concrete_legro_2025-06-09-14-23-57.bag", index=False)


    print(f"Contact state data shape: {raw_contact_df.shape}")

    # Display column information
    print("\n=== CONTACT STATE DATA COLUMNS ===")
    print(raw_contact_df.columns.tolist())
    print("\nFirst few rows of contact state data:")
    print(raw_contact_df.head())

    return raw_contact_df

def plot_contact_state(raw_contact_df, time_window=None):
    """
    Plot only the raw contact state (ee_contact) over time for 4 legs
    """
    if raw_contact_df is None:
        print("No data to plot!")
        return

    # Convert timestamps if they exist
    if 'Time' in raw_contact_df.columns:
        raw_contact_df['timestamp'] = pd.to_numeric(raw_contact_df['Time'])
        raw_contact_df['time_sec'] = raw_contact_df['timestamp'] - raw_contact_df['timestamp'].min()

    # Apply time window if specified
    if time_window:
        start_time, end_time = time_window
        raw_contact_df = raw_contact_df[(raw_contact_df['time_sec'] >= start_time) &
                                        (raw_contact_df['time_sec'] <= end_time)]

    # Expand ee_contact into separate columns
    ee_array = raw_contact_df['ee_contact'].apply(lambda x: eval(x))  # convert string list to list
    ee_df = pd.DataFrame(ee_array.tolist(), columns=['FL', 'FR', 'BL', 'BR'])
    raw_contact_df = pd.concat([raw_contact_df, ee_df], axis=1)

    legs = ['FL', 'FR', 'BL', 'BR']

    # Create separate subplots
    fig, axes = plt.subplots(4, 1, figsize=(15, 10), sharex=True)
    fig.suptitle('Raw Contact States Over Time (Separate Legs)', fontsize=16, fontweight='bold')

    for i, leg in enumerate(legs):
        axes[i].plot(raw_contact_df['time_sec'], raw_contact_df[leg], color='blue', linewidth=2)
        axes[i].set_ylabel(f'{leg} Contact')
        axes[i].set_ylim(-0.1, 1.1)
        axes[i].grid(True, alpha=0.3)

    axes[-1].set_xlabel('Time (seconds)')
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# Main execution function
def main(bag_file, time_window=None):
    """
    Main function to analyze contact data
    """
    print("Starting contact data analysis...")
    print(f"Bag file: {bag_file}")

    # Convert and read data
    contact_df = analyze_contact_data(bag_file)

    if contact_df is not None:
        print("\n" + "="*50)
        print("Creating visualizations...")
        plot_contact_state(contact_df, time_window)
    else:
        print("Failed to read contact data!")

# Example usage
if __name__ == "__main__":
    bag_file = "concrete_legro_2025-06-09-14-23-57.bag"
    main(bag_file)

    # Or analyze specific time window: main(bag_file, time_window=(10, 60))


In [ ]:
import os
from bagpy import bagreader
import pandas as pd
import numpy as np
import re
from scipy.spatial.transform import Rotation as R

def extract_values(string_data):
    pattern = r'[-+]?\d*\.\d+(?:[eE][-+]?\d+)?|[-+]?\d+[eE][-+]?\d+|[-+]?\d+'
    numbers = re.findall(pattern, string_data)
    return np.array([float(num) for num in numbers], dtype=float)

def is_in_body_frame(ee_array):
    flat = ee_array.flatten()
    return np.all(np.abs(flat) < 5.0)

bag_paths = [
    #"./data/dataset_path",
    #"./data/dataset_path",
    #"./data/dataset_path",
    #"./data/dataset_path",
    #"./data/dataset_path",
    #"./data/dataset_path",
    #"./data/dataset_path",
    #"./data/dataset_path",
    #"./data/dataset_path",
    #"./data/dataset_path"
    "concrete_legro_2025-06-09-14-23-57.bag"
]

output_csv_dir = "./data/dataset_path"
os.makedirs(output_csv_dir, exist_ok=True)

for bag_path in bag_paths:
    print(f"\n Processing {bag_path}...")

    try:
        b = bagreader(bag_path)
        state_csv_path = b.message_by_topic("/estimated_robot_state")
        imu_csv = b.message_by_topic("/unitree/imu")

        state_df = pd.read_csv(state_csv_path)
        imu_df = pd.read_csv(imu_csv)

        # Extract values from columns
        state_df['ee_pos'] = state_df['ee_pos'].apply(lambda x: extract_values(str(x)))
        state_df['ee_vel'] = state_df['ee_vel'].apply(lambda x: extract_values(str(x)))
        state_df['ee_forces_cleaned'] = state_df['ee_forces'].apply(lambda x: extract_values(str(x)))

        # Frame check
        first_ee = state_df['ee_pos'].iloc[0]
        if is_in_body_frame(first_ee):
            print("Foot positions/velocities already in body frame → skipping conversion.")
            state_df['ee_pos_cleaned'] = state_df['ee_pos']
            state_df['ee_vel_cleaned'] = state_df['ee_vel']
            state_df['ee_vel_frame'] = "body"
        else:
            print("Converting foot positions and velocities from world to body frame...")
            base_pos = state_df[['base_pose.position.x', 'base_pose.position.y', 'base_pose.position.z']].to_numpy()
            base_quat = state_df[['base_pose.orientation.x', 'base_pose.orientation.y', 'base_pose.orientation.z', 'base_pose.orientation.w']].to_numpy()

            ee_pos_world = np.vstack(state_df['ee_pos'].to_numpy())
            ee_vel_world = np.vstack(state_df['ee_vel'].to_numpy())
            r_all = R.from_quat(base_quat)

            ee_pos_body = np.zeros_like(ee_pos_world)
            ee_vel_body = np.zeros_like(ee_vel_world)

            for i in range(4):  # 4 feet
                pos_diff = ee_pos_world[:, i*3:(i+1)*3] - base_pos
                ee_pos_body[:, i*3:(i+1)*3] = r_all.inv().apply(pos_diff)
                ee_vel_body[:, i*3:(i+1)*3] = r_all.inv().apply(ee_vel_world[:, i*3:(i+1)*3])

            state_df['ee_pos_cleaned'] = list(ee_pos_body)
            state_df['ee_vel_cleaned'] = list(ee_vel_body)
            state_df['ee_vel_frame'] = "body"

        # Remove unused column
        state_df.drop(columns=['ee_acc'], inplace=True, errors='ignore')

        # Save CSVs
        base_name = os.path.splitext(os.path.basename(bag_path))[0]
        output_csv_path = os.path.join(output_csv_dir, f"{base_name}_cleaned.csv")
        state_df.to_csv(output_csv_path, index=False)
        print(f"Saved cleaned CSV: {output_csv_path}")

        imu_output_csv_path = os.path.join(output_csv_dir, f"{base_name}_imu.csv")
        imu_df.to_csv(imu_output_csv_path, index=False)
        print(f"Saved IMU CSV: {imu_output_csv_path}")

    except Exception as e:
        print(f"Failed to process {bag_path} because: {e}")

In [ ]:
import os
import numpy as np

# Config Paths
preprocessed_dir = "./data/dataset_path"
final_output = "./data/dataset_path"

os.makedirs(final_output, exist_ok=True)

print("\n--- Building final datasets ---")

# List terrain folders inside preprocessed_dir
terrain_dirs = sorted([d for d in os.listdir(preprocessed_dir) if os.path.isdir(os.path.join(preprocessed_dir, d))])
print(f"Found {len(terrain_dirs)} terrain datasets.")

print("\n--- Building final datasets ---")

# List terrain folders inside preprocessed_dir
terrain_dirs = sorted([d for d in os.listdir(preprocessed_dir) if os.path.isdir(os.path.join(preprocessed_dir, d))])
print(f"Found {len(terrain_dirs)} terrain datasets.")

for terrain in terrain_dirs:
    try:
        print(f"\nProcessing terrain: {terrain}")

        terrain_path = os.path.join(preprocessed_dir, terrain)

        # Load data files
        rq = np.load(os.path.join(terrain_path, 'q.npy'))
        rqd = np.load(os.path.join(terrain_path, 'qd.npy'))
        rp = np.load(os.path.join(terrain_path, 'p.npy'))
        rv = np.load(os.path.join(terrain_path, 'v.npy'))
        acc = np.load(os.path.join(terrain_path, 'acc.npy'))
        omega = np.load(os.path.join(terrain_path, 'omega.npy'))

        # Load contacts
        contacts = np.load(os.path.join(terrain_path, 'contacts.npy'))

        # Load labels if exist
        labels_path = os.path.join(terrain_path, 'contacts_labels.npy')
        if os.path.exists(labels_path):
            labels = np.load(labels_path)
        else:
            labels = np.zeros(rq.shape[0], dtype=int)

        # Concatenate horizontally along features axis (axis=1)
        data = np.concatenate((rq, rqd, rp, rv, acc, omega), axis=1)

        # Save concatenated data and labels in final output folder
        out_dir = os.path.join(final_output, terrain)
        os.makedirs(out_dir, exist_ok=True)

        np.save(os.path.join(out_dir, 'data_reordered.npy'), data)
        np.save(os.path.join(out_dir, 'labels_reordered.npy'), labels)

        print(f"Saved data shape: {data.shape}, labels shape: {labels.shape}")
        print(f"Data_type: {data.dtype}, Labels_type: {labels.dtype}")
        print(f"Data size (MB): {data.nbytes / (1024*1024):.2f} MB")
    except Exception as e:
        print(f"Error processing terrain '{terrain}': {e}")
        
# This script builds the final model-ready datasets from preprocessed terrain folders.

# For each terrain folder in `preprocessed_data/`:
# - Loads individual numpy arrays: q (positions), qd (velocities), p (foot pos), v (foot vel)
# - Loads contact labels (`contacts_labels.npy`) or defaults to zeros if missing
# - Concatenates q, qd, p, v into a single input matrix `data` with shape (N, 48)
# - Saves:
#   - `data_reordered.npy` → the full input features
#   - `labels_reordered.npy` → the corresponding contact labels
# - Outputs are organized under `preprocessed_data/final/<terrain>/`


COMBINING EVERYTHING 

In [ ]:
import os
import numpy as np

# Optional: fix the random seed for reproducibility (not needed anymore but kept if you want)
#np.random.seed(42)

# Directory to save all split files
save_dir = "./data/dataset_path"
os.makedirs(save_dir, exist_ok=True)

# Define all terrains to process
terrains = {
    "asphalt": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")),
    "concrete": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")),
    "GRASS": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")),
    "GRAVEL": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")),
    "LONGWALKROADUSP": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")),
    "RAMP": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")),
    "SIDEWALK": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")),
    "smallpebbles": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path"))
}

# save as train all data (X_train, Y_train)

# Loop over terrains and process splits
for terrain_name, (data, labels) in terrains.items():
    assert data.shape[0] == labels.shape[0], f"Mismatch in {terrain_name}"

    n = data.shape[0]

    # No shuffling here — split data in original order
    train_end = int(n)
    #val_end = int(0.85 * n)

    X_train = data[:train_end]
    Y_train = labels[:train_end]

    #X_val = data[train_end:val_end]
    #Y_val = labels[train_end:val_end]

    #X_test = data[val_end:]
    #Y_test = labels[val_end:]

    # Save splits to the new directory
    np.save(os.path.join(save_dir, f"X_train_{terrain_name}.npy"), X_train)
    np.save(os.path.join(save_dir, f"Y_train_{terrain_name}.npy"), Y_train)
    #np.save(os.path.join(save_dir, f"X_val_{terrain_name}.npy"), X_val)
    #np.save(os.path.join(save_dir, f"Y_val_{terrain_name}.npy"), Y_val)
    #np.save(os.path.join(save_dir, f"X_test_{terrain_name}.npy"), X_test)
    #np.save(os.path.join(save_dir, f"Y_test_{terrain_name}.npy"), Y_test)

    # Print summary
    print(f"{terrain_name} Splits:")
    print(f"  Train -> X: {X_train.shape}, Y: {Y_train.shape}")
    #print(f"  Val   -> X: {X_val.shape}, Y: {Y_val.shape}")
    #print(f"  Test  -> X: {X_test.shape}, Y: {Y_test.shape}")
    print("Saved splits successfully.\n")


In [ ]:
import os
import numpy as np

root_dir = "./data/dataset_path"
save_dir = "./data/dataset_path"
os.makedirs(save_dir, exist_ok=True)

# List all files in the root directory
all_files = os.listdir(root_dir)

# Filter files for train data (X and Y)
x_train_files = sorted([f for f in all_files if f.startswith("X_train_") and f.endswith(".npy")])
y_train_files = sorted([f for f in all_files if f.startswith("Y_train_") and f.endswith(".npy")])

# Ensure we have the same terrains in both X and Y
terrains = set(f.split("X_train_")[1].split(".npy")[0] for f in x_train_files)

# Load and concatenate all train X data
X_train_list = [np.load(os.path.join(root_dir, f)) for f in x_train_files]
X_train = np.concatenate(X_train_list, axis=0)

# Load and concatenate all train Y data
Y_train_list = [np.load(os.path.join(root_dir, f)) for f in y_train_files]
Y_train = np.concatenate(Y_train_list, axis=0)

print(f"Concatenated X_train shape: {X_train.shape}")
print(f"Concatenated Y_train shape: {Y_train.shape}")

# Save concatenated arrays
np.save(os.path.join(save_dir, "X_train.npy"), X_train)
np.save(os.path.join(save_dir, "Y_train.npy"), Y_train)

print(f"Saved concatenated train splits to {save_dir}")



*** the end **
HOW TO GET THE NOISE JSON FILES 

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt
from scipy.optimize import curve_fit

def lowpass_filter(data, cutoff=5, fs=200, order=2):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data, axis=0)

def estimate_bias_and_std(signal, smoothed_signal):
    noise = signal - smoothed_signal
    bias = np.mean(noise, axis=0)
    std = np.std(noise, axis=0)
    return bias, std

def ou_autocorr(t, sigma, tau):
    return sigma**2 * np.exp(-t/tau)

def estimate_ou_params(signal, dt=1/200):
    # Remove mean
    signal = signal - np.mean(signal)

    # Compute autocorrelation (biased)
    acf = np.correlate(signal, signal, mode='full')[len(signal)-1:]
    acf /= acf[0]  # normalize

    # Time vector for autocorrelation
    t = np.arange(len(acf)) * dt

    # Fit OU autocorrelation function to ACF (only first 50 points to avoid noise)
    try:
        popt, _ = curve_fit(ou_autocorr, t[:50], acf[:50], bounds=(0, np.inf))
        sigma_est, tau_est = popt
    except Exception as e:
        print(f"OU fit failed: {e}")
        sigma_est, tau_est = 0.0, 0.0

    return sigma_est, tau_est

def analyze_imu_noise(acc, omega, dt=1/200, fs=200):
    # Smooth acceleration to get underlying signal
    acc_smooth = lowpass_filter(acc, cutoff=5, fs=fs, order=2)
    acc_bias, acc_white_std = estimate_bias_and_std(acc, acc_smooth)

    # Weight of lowpass noise in total acc noise (estimate)
    lowpass_noise = acc_smooth - np.mean(acc_smooth, axis=0)
    lowpass_weight = np.linalg.norm(lowpass_noise) / (np.linalg.norm(acc - np.mean(acc, axis=0)) + 1e-6)

    # Angular velocity noise residuals
    omega_bias = np.mean(omega, axis=0)
    omega_residuals = omega - omega_bias - lowpass_filter(omega, cutoff=5, fs=fs, order=2)

    # OU parameters for each axis
    sigma_vals = []
    tau_vals = []
    for i in range(3):
        sigma_i, tau_i = estimate_ou_params(omega_residuals[:, i], dt)
        sigma_vals.append(sigma_i)
        tau_vals.append(tau_i)

    # Estimate gyro white noise std as std of high-frequency residuals
    gyro_white_std = np.std(omega_residuals - lowpass_filter(omega_residuals, cutoff=20, fs=fs, order=2), axis=0)

    # Package results
    noise_profile = {
        "acc_white_std": acc_white_std.tolist(),
        "acc_bias": acc_bias.tolist(),
        "acc_lowpass_weight": lowpass_weight,
        "omega_colored_sigma": sigma_vals,
        "omega_colored_tau": tau_vals,
        "gyro_white_std": gyro_white_std.tolist(),
        "gyro_bias": omega_bias.tolist(),
        "sampling_dt": dt,
        "colored_noise_type": "Ornstein-Uhlenbeck"
    }
    return noise_profile

# Usage Example:
if __name__ == "__main__":
    import json

    # Load your asphalt IMU CSV (replace path and column names as needed)
    file_path = "./content/realdata/cleaned_csvs/asphalt_legro_2025-06-09-14-12-15_imu.csv"
    df = pd.read_csv(file_path)

    acc = df[['linear_acceleration.x', 'linear_acceleration.y', 'linear_acceleration.z']].to_numpy()
    omega = df[['angular_velocity.x', 'angular_velocity.y', 'angular_velocity.z']].to_numpy()

    noise_params = analyze_imu_noise(acc, omega, dt=1/200, fs=200)

    print("Derived Noise Parameters:")
    print(json.dumps(noise_params, indent=2))

    # Optionally save to file
    with open("asphalt_noise_profile.json", "w") as f:
        json.dump(noise_params, f, indent=2)
        
  
